In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules

In [0]:
%pip install pyyaml

## Configure Parameters

**IMPORTANT**: Modify these parameters for your environment before running!

In [0]:
from databricks.sdk import WorkspaceClient
from utilities import load_input_csv
from salesforce.connector import SalesforceConnector

w = WorkspaceClient()

# Automatically get current user email
username = w.current_user.me().user_name

# Automatically get workspace host URL
workspace_host = w.config.host

# Automatically construct output directory
WORKSPACE_HOST = workspace_host
ROOT_PATH = f'/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}'



In [0]:
from core.runner import run_pipeline_generation, list_connectors, get_connector_info

connector_name = "sf"

input_csv = 'examples/tapworks/pipeline_config.csv'
output_dir = f'/Workspace/Users/{username}/lakehouse-tapworks/salesforce/examples/tapworks/deployment'

# df = load_input_csv(INPUT_CSV)

targets={
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
}

max_tables_per_pipeline = 100
max_tables_per_gateway = 100

# Run pipeline generation
result_df = run_pipeline_generation(
    connector_name=connector_name,
    input_source=input_csv,
    output_dir=output_dir,
    targets=targets,
    max_tables_per_pipeline=max_tables_per_pipeline,
    max_tables_per_gateway=max_tables_per_gateway,
)

print(f"\nProcessed {len(result_df)} tables")
print(f"Pipeline groups: {result_df['pipeline_group'].nunique()}")
if 'gateway' in result_df.columns:
    print(f"Gateways: {result_df['gateway'].nunique()}")

# # Create connector
# connector = SalesforceConnector()

# # Generate pipelines
# result = connector.run_complete_pipeline_generation(
# df=df,
# output_dir=OUTPUT_DIR,
# targets={
#     'dev': {
#         'workspace_host': WORKSPACE_HOST,
#         'root_path': ROOT_PATH
#     }
# },
# max_tables_per_pipeline=250
# )

## Example: Using Custom Default Values and Overrides

You can use `default_values` and `override_input_config` parameters for advanced configuration:

**`default_values` Parameter:**
- Provides custom default values for optional columns
- Merged with built-in defaults (your values take precedence)
- Applied only to missing/empty values in the CSV

**`override_input_config` Parameter:**
- Forces specific column values for ALL rows
- Overwrites any existing values in the CSV
- Useful for environment-specific overrides

**Example Use Cases:**
- Override schedule for testing (e.g., disable auto-runs)
- Force all pipelines to use a specific catalog/schema for dev environment
- Set custom defaults for columns not in your CSV

In [0]:
INPUT_CSV = 'examples/tapworks/pipeline_config.csv'
OUTPUT_DIR = f'/Workspace/Users/{username}/lakehouse-tapworks/salesforce/examples/tapworks/deployment'

df = load_input_csv(INPUT_CSV)

# Create connector
connector = SalesforceConnector()

# Generate pipelines
result = connector.run_complete_pipeline_generation(
df=df,
output_dir=OUTPUT_DIR,
targets={
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
},
max_tables_per_pipeline=250,
override_input_config={
    'target_catalog': 'dev_catalog',
    'schedule': '0 0 * * 0'}
)

## Understanding the Input CSV

Your CSV should have the following columns:

**Required Columns:**
- `source_database`: Usually "Salesforce"
- `source_schema`: "standard" or "custom"
- `source_table_name`: Salesforce object name (e.g., "Account", "Contact")
- `target_catalog`: Target Databricks catalog
- `target_schema`: Target Databricks schema
- `target_table_name`: Destination table name
- `prefix`: Business unit or logical grouping (e.g., "business_unit1")
- `subgroup`: subgroup level (e.g., "01", "02")

**Optional Columns:**
- `connection_name`: Salesforce connection name (uses default if not specified)
- `schedule`: Cron schedule (uses default if not specified)
- `include_columns`: Comma-separated list of columns to include
- `exclude_columns`: Comma-separated list of columns to exclude

**Pipeline Grouping Logic:**
- Objects are grouped by unique combinations of (prefix, subgroup)
- Each group becomes a separate pipeline
- Example: All objects with prefix="business_unit1" and subgroup="01" go into one pipeline